# NeMo Fraud Detection – Kompatibilitätsfassung für `nvcr.io/nvidia/nemo:24.09`

**Wichtiger Kompatibilitätshinweis:** NeMo Framework 24.09 basiert auf NeMo 2.0 (u. a. PyTorch 2.4.0a0, Megatron Core 0.9.0). Das ursprüngliche Notebook verwendet dagegen die NeMo-1.x-CLI `megatron_gpt_finetuning.py` und einen alten/abweichenden Konverter-Aufruf. Diese beiden Workflows dürfen nicht einfach vermischt werden.

Die archivierte 24.09-Dokumentation beschreibt für die NeMo-2.0-Llama-Rezepte Llama 3 8B und 70B, aber kein fertiges Rezept für **Llama 3.2 3B Instruct**. Daher prüft dieses Notebook die tatsächlich im Container vorhandenen APIs und bricht vor einem inkompatiblen Training mit einer aussagekräftigen Meldung ab, statt versehentlich eine 8B-Konfiguration auf 3B-Gewichte anzuwenden.

Die übrigen Teile (Container-/GPU-Prüfung, Datenprüfung, Pfadverwaltung, echte Inferenz-Evaluation und optionales W&B-Logging) sind bereinigt. Für echtes Megatron-SFT von Llama 3.2 3B ist entweder ein explizit unterstütztes NeMo-24.09-Modell-/Config-Mapping oder ein neuerer NeMo-Stack mit dokumentierter Llama-3.2-3B-Unterstützung erforderlich.

## 0. Container starten

Beispiel auf dem Host (Datenpfade bei Bedarf anpassen):

```bash
docker pull nvcr.io/nvidia/nemo:24.09

docker run --rm -it --gpus all --network host --shm-size=16g \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  -v "$PWD":/workspace -v /data:/data \
  -w /workspace nvcr.io/nvidia/nemo:24.09
```

Notebook im Container starten. Keine PyTorch-/CUDA-Pakete per `pip` überinstallieren: Der Container bringt zueinander passende NVIDIA-Builds mit.

In [13]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
import torch

# Zentrale Konfiguration – nur diese Pfade/Parameter bei Bedarf anpassen.
DATA_ROOT = Path("/data/nemo-fraud-detection-notebooks/notebooks")
HF_MODEL_DIR = Path("/data/huggingface/llama-3.2-3b-instruct")
TRAIN_FILE = DATA_ROOT / "02_Data_Curation/data/sft/train.jsonl"
VAL_FILE = DATA_ROOT / "02_Data_Curation/data/sft/validation.jsonl"

WORK_DIR = Path("/workspace")
OUTPUT_DIR = WORK_DIR / "outputs"
NEMO_CACHE_DIR = OUTPUT_DIR / "nemo_cache"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
HF_EXPORT_DIR = OUTPUT_DIR / "final_model_hf"

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
MAX_STEPS = 10
MAX_NEW_TOKENS = 24
USE_WANDB = False
WANDB_PROJECT = "nemo-fraud-detection"

for p in (OUTPUT_DIR, NEMO_CACHE_DIR, CHECKPOINT_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("NeMo 24.09 Container erwartet.")

Python: 3.10.12
PyTorch: 2.5.1+cu121
CUDA runtime: 12.1
NeMo 24.09 Container erwartet.


## 1. Container-, GPU- und Speicherprüfung

Die Swap-Prüfung ist rein informativ. Ein Container kann Swap des Hosts nicht zuverlässig selbst einrichten; das muss außerhalb des Containers erfolgen.

In [14]:
def check_environment():
    try:
        import nemo
        nemo_version = getattr(nemo, "__version__", "unbekannt")
    except Exception as exc:
        raise RuntimeError("NeMo konnte nicht importiert werden. Läuft das Notebook im NeMo-Container?") from exc

    print("NeMo:", nemo_version)
    print("PyTorch:", torch.__version__)

    if not torch.cuda.is_available():
        raise RuntimeError("Keine CUDA-GPU sichtbar. Container mit --gpus all starten.")

    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name} – {props.total_memory / 1024**3:.1f} GiB VRAM")

    total, used, free = shutil.disk_usage(str(OUTPUT_DIR))
    print(f"Freier Speicher auf Output-Dateisystem: {free / 1024**3:.1f} GiB")

    meminfo = Path("/proc/meminfo").read_text()
    swap_line = next((x for x in meminfo.splitlines() if x.startswith("SwapTotal:")), None)
    print("Swap:", swap_line or "nicht verfügbar")

    if free < 20 * 1024**3:
        raise RuntimeError("Weniger als 20 GiB freier Speicher im Output-Dateisystem.")

check_environment()

NeMo: 2.0.0
PyTorch: 2.5.1+cu121
GPU 0: NVIDIA L4 – 22.0 GiB VRAM
Freier Speicher auf Output-Dateisystem: 517.2 GiB
Swap: SwapTotal:             0 kB


## 2. Datenformat validieren

Erwartet wird JSONL mit den Feldern `input` und `output`. Die Prüfung liest die Dateien ein, verändert sie aber nicht.

In [15]:
def validate_jsonl(path: Path, required=("input", "output")):
    if not path.is_file():
        raise FileNotFoundError(f"Datei nicht gefunden: {path}")
    count = 0
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"{path}:{line_no}: ungültiges JSON: {exc}") from exc
            missing = [k for k in required if k not in row]
            if missing:
                raise ValueError(f"{path}:{line_no}: fehlende Felder {missing}")
            if not isinstance(row["input"], str) or not isinstance(row["output"], str):
                raise TypeError(f"{path}:{line_no}: input/output müssen Strings sein")
            count += 1
    if count == 0:
        raise ValueError(f"Keine Datensätze in {path}")
    print(f"{path}: {count} Datensätze")
    return count

n_train = validate_jsonl(TRAIN_FILE)
n_val = validate_jsonl(VAL_FILE)

/data/nemo-fraud-detection-notebooks/notebooks/02_Data_Curation/data/sft/train.jsonl: 54 Datensätze
/data/nemo-fraud-detection-notebooks/notebooks/02_Data_Curation/data/sft/validation.jsonl: 11 Datensätze


## 3. NeMo-2.0-Kompatibilität für Llama 3.2 3B feststellen

Der ursprüngliche Aufruf

```text
/opt/NeMo/examples/nlp/language_modeling/tuning/megatron_gpt_finetuning.py
```

gehört zum NeMo-1.x-SFT-Workflow. NeMo 2.0 dokumentiert stattdessen `nemo.collections.llm`, `LlamaModel`, `FineTuningDataModule` und `Trainer.fit(...)`.

Die folgende Zelle prüft, ob die installierte 24.09-Version eine passende 3B-Konfiguration bereitstellt. **Sie verwendet ausdrücklich nicht ersatzweise `Llama3Config8B`.**

In [16]:
def inspect_nemo2_llama_support():
    try:
        from nemo.collections import llm
    except Exception as exc:
        raise RuntimeError(
            "nemo.collections.llm konnte nicht importiert werden. "
            "Dies ist nicht die erwartete NeMo-2.0-Umgebung."
        ) from exc

    candidates = [
        name for name in dir(llm)
        if ("Llama" in name or "llama" in name) and ("Config" in name or "Model" in name)
    ]
    print("Llama-bezogene Symbole in nemo.collections.llm:")
    print("\n".join(sorted(candidates)) if candidates else "(keine gefunden)")

    # Die 24.09-Dokumentation führt Llama-3-Rezepte für 8B/70B auf.
    # Ein explizites 3B-Config-Mapping ist Voraussetzung, nicht optional.
    possible = [
        "Llama32Config3B", "Llama3_2Config3B", "Llama3Config3B",
        "Llama3_2_3BConfig", "Llama3_2_3B"
    ]
    found = [name for name in possible if hasattr(llm, name)]
    print("Explizite 3B-Konfigurationen:", found or "keine")

    return llm, found

llm, llama3b_config_names = inspect_nemo2_llama_support()

2026-11-08 20:38:17.961606272 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
[NeMo W 2026-11-08 20:38:18 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
      from .autonotebook import tqdm as notebook_tqdm
    
[NeMo W 2026-11-08 20:38:22 nemo_logging:349] /opt/megatron-lm/megatron/core/tensor_parallel/layers.py:280: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
      def forward(ctx, input, weight, bias, allreduce_dgrad):
    
[NeMo W 2026-11-08 20:38:22 nemo_logging:349] /opt/megatron-lm/megatron/core/tensor_parallel/layers.py:290: FutureWarning: `torch.cuda.amp.custom_bwd(arg

RuntimeError: nemo.collections.llm konnte nicht importiert werden. Dies ist nicht die erwartete NeMo-2.0-Umgebung.

## 4. Fine-Tuning (sicherer Kompatibilitäts-Gate)

Wenn der Container eine passende 3B-Konfiguration bereitstellt, muss deren Modell- und Datamodule-Mapping gegen die konkrete NeMo-24.09-Implementierung validiert werden. Die offizielle 24.09-Llama-Seite zeigt als fertige Recipes Llama 3 8B/70B; ein Llama-3.2-3B-Rezept ist dort nicht dokumentiert.

Diese Zelle stoppt daher standardmäßig. Das ist beabsichtigt: Ein automatischer Rückfall auf 8B wäre fachlich falsch und könnte Checkpoints beschädigen oder zu Shape-Fehlern führen.

In [ ]:
def run_megatron_finetuning():
    if not llama3b_config_names:
        raise NotImplementedError(
            "Kein explizites Llama-3.2-3B-Config-Mapping in dieser NeMo-24.09-Installation gefunden. "
            "Die dokumentierten NeMo-2.0-Llama-Rezepte dieser Version decken Llama 3 8B/70B ab, "
            "nicht Llama 3.2 3B. Das ursprüngliche megatron_gpt_finetuning.py ist der NeMo-1.x-"
            "Workflow und kein kompatibler Ersatz für den NeMo-2.0-Workflow. "
            "Bitte neuere NeMo-Version mit Llama-3.2-3B-Unterstützung verwenden oder ein geprüftes "
            "3B-ModelConfig/DataModule-Mapping ergänzen."
        )

    raise NotImplementedError(
        "Eine 3B-Klasse ist vorhanden, aber ihr konkretes Modell-/Datamodule-Mapping wurde "
        "nicht automatisch geraten. Bitte Mapping anhand der installierten NeMo-24.09-API "
        "explizit konfigurieren, bevor Training gestartet wird."
    )

# Explizit aktivieren, wenn der obige Kompatibilitäts-Gate erfolgreich ergänzt wurde:
RUN_TRAINING = False
if RUN_TRAINING:
    run_megatron_finetuning()
else:
    print("Training nicht gestartet. RUN_TRAINING=False; siehe Kompatibilitätshinweis.")

## 5. Echte lokale Inferenz-Evaluation

Die Originalfassung setzte `prediction = expected` und erzeugte damit einen perfekten, aber bedeutungslosen Klassifikationsreport. Diese Fassung generiert tatsächliche Modellantworten. Voraussetzung ist ein HF-kompatibler Export unter `HF_EXPORT_DIR`.

Hinweis: Falls das trainierte Modell ein LoRA-Adapter ist, muss der Adapter vor dem Laden mit dem Basismodell zusammengeführt oder korrekt als PEFT-Modell geladen werden.

In [ ]:
def load_hf_model(model_dir: Path):
    if not model_dir.is_dir():
        raise FileNotFoundError(
            f"HF-Modellverzeichnis fehlt: {model_dir}. "
            "Ein NeMo-Checkpoint (.ckpt/.nemo) ist nicht automatisch ein HF-Verzeichnis."
        )
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        str(model_dir),
        torch_dtype=dtype,
        device_map="auto",
    )
    model.eval()
    return model, tokenizer

def generate_prediction(model, tokenizer, prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def evaluate_hf_model(model_dir: Path, data_path: Path):
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

    model, tokenizer = load_hf_model(model_dir)
    y_true, y_pred, rows = [], [], []

    with data_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            expected = item["output"].strip()
            predicted = generate_prediction(model, tokenizer, item["input"]).strip()
            y_true.append(expected)
            y_pred.append(predicted)
            rows.append({"input": item["input"], "expected": expected, "predicted": predicted})

    print(classification_report(y_true, y_pred, zero_division=0))
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    return rows, y_true, y_pred

# Erst nach einem gültigen HF-Export aktivieren:
RUN_EVALUATION = False
if RUN_EVALUATION:
    evaluation_rows, y_true, y_pred = evaluate_hf_model(HF_EXPORT_DIR, VAL_FILE)
else:
    print("Evaluation übersprungen: RUN_EVALUATION=False oder HF-Export noch nicht vorhanden.")

## 6. Optionales W&B-Logging

W&B ist optional. Es werden nur echte Inferenzresultate geloggt, keine künstlich perfekten Fallback-Vorhersagen.

In [ ]:
def log_evaluation_to_wandb(rows, y_true, y_pred):
    import wandb
    from sklearn.metrics import accuracy_score, f1_score

    run = wandb.init(project=WANDB_PROJECT, name="llama32-3b-evaluation", job_type="evaluation")
    try:
        table = wandb.Table(columns=["input_text", "expected_label", "predicted_label"])
        for row in rows:
            table.add_data(row["input"], row["expected"], row["predicted"])

        wandb.log({
            "eval/accuracy": accuracy_score(y_true, y_pred),
            "eval/macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "evaluation_table": table,
        })
    finally:
        run.finish()

if USE_WANDB and RUN_EVALUATION:
    log_evaluation_to_wandb(evaluation_rows, y_true, y_pred)
else:
    print("W&B-Logging übersprungen.")

## 7. Was für ein vollständig ausführbares Megatron-Training noch fehlt

Für die gewünschte Kombination **NeMo 24.09 + Llama 3.2 3B Instruct + Megatron-SFT** ist ein verifiziertes 3B-Modellmapping nötig. In dieser Notebook-Version wird das nicht erfunden.

Nachgewiesene Unterschiede zum Original:
- Der Container `nvcr.io/nvidia/nemo:24.09` gehört zur NeMo-2.0-Linie.
- NeMo 2.0 verwendet für SFT `Trainer.fit(model, data, ...)` bzw. NeMo-Run und nicht den alten NeMo-1.x-CLI-Aufruf.
- Die 24.09-Llama-Dokumentation nennt Rezepte für Llama 3 8B und 70B, nicht Llama 3.2 3B.
- Der bisherige Exportbefehl `nemo llm export model=auto ...` wird entfernt; er ist kein verlässlicher Exportpfad für den hier verwendeten alten Checkpoint.
- Der bisherige Evaluierungs-Fallback `prediction = expected` wurde durch echte Generierung ersetzt.

Offizielle Referenzen:
- [NeMo 24.09 Installation](https://docs.nvidia.com/nemo-framework/user-guide/24.09/installation.html)
- [NeMo 24.09 Software-Komponenten](https://docs.nvidia.com/nemo-framework/user-guide/24.09/softwarecomponentversions.html)
- [NeMo 2.0 SFT-Migration](https://docs.nvidia.com/nemo-framework/user-guide/24.09/nemo-2.0/migration/sft.html)
- [NeMo 24.09 Llama-Rezepte](https://docs.nvidia.com/nemo-framework/user-guide/24.09/llms/llama.html)